# Concrete Compressive Strength — Machine Learning & Data Analytics
**Module:** 6COSC017C-n — Machine Learning and Data Analytics  
**Dataset:** Concrete Compressive Strength (UCI ML Repository)  
**Source:** https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength  
**Task type:** Regression  
**Deadline:** June 10, 2026

---
## A. Introduction

**Business / Scientific Case:**  
Concrete is the most widely used construction material in the world. Its compressive strength (measured in MPa — megapascals) is the single most important mechanical property that determines structural safety. Traditionally, measuring concrete strength requires casting samples and waiting 28 days for them to cure before testing. This delay is costly and slows construction timelines.

Machine learning offers an alternative: by knowing the mix proportions and age, we can **predict compressive strength without waiting**, enabling engineers to optimise mix designs faster and more cheaply.

**Dataset Origin & Licensing:**  
The dataset was compiled by Prof. I-Cheng Yeh (Chung-Hua University, Taiwan) and is publicly available through the UCI Machine Learning Repository under a CC BY 4.0 licence.  
- **URL:** https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength  
- **Original paper:** Yeh, I.C. (1998). *Modeling of strength of high-performance concrete using artificial neural networks*. Cement and Concrete Research, 28(12), 1797–1808.

**Problem Framing:**  
This is a **supervised regression** problem. We predict the continuous target variable — concrete compressive strength (MPa) — from eight quantitative input features describing the concrete mix and curing age.

---
## Setup — Import Libraries

In [ ]:
# Suppress non-critical warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

# Core data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical testing (Q-Q plots, Z-scores)
from scipy import stats

# Scikit-learn: model selection and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Scikit-learn: regression algorithms
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Scikit-learn: evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set global random seed for full reproducibility across all algorithms
SEED = 42
np.random.seed(SEED)

# Global plot style settings
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

print('All libraries imported successfully.')

---
## A. Load Dataset

In [ ]:
# Load the dataset directly from the UCI ML Repository (Excel format)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls'
df = pd.read_excel(url)

# Rename columns from verbose UCI names to short, readable identifiers
# All ingredient quantities are in kg per cubic metre of concrete (kg/m³)
df.columns = [
    'cement',              # Cement content (kg/m³)
    'blast_furnace_slag',  # Blast furnace slag — a cement substitute (kg/m³)
    'fly_ash',             # Fly ash — a cement substitute (kg/m³)
    'water',               # Water content (kg/m³)
    'superplasticizer',    # Chemical admixture that improves workability (kg/m³)
    'coarse_aggregate',    # Coarse aggregate — gravel or crushed stone (kg/m³)
    'fine_aggregate',      # Fine aggregate — sand (kg/m³)
    'age',                 # Curing age in days (1–365)
    'strength'             # TARGET: compressive strength in MPa
]

# Save a local CSV copy so the notebook can be re-run without internet access
df.to_csv('Concrete_Data.csv', index=False)

print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

---
## B. Exploratory Data Analysis (EDA)

### B1. Dataset Shape and Data Types

In [ ]:
# Check overall dimensions of the dataset
print(f'Shape: {df.shape}')

# Confirm all columns are numeric — required for ML algorithms without encoding
print(f'\nData types:')
print(df.dtypes)

# Check for missing values in each column
# Zero values in slag, fly_ash, superplasticizer are VALID (optional ingredients)
print(f'\nMissing values per column:')
print(df.isnull().sum())

**Justification:** Checking shape and dtypes confirms all features are numeric (float64/int64), which is expected for a physical mixture dataset. Identifying missing values early prevents silent errors during modelling.

### B2. Summary Statistics (Measures of Central Tendency)

In [ ]:
# Generate standard descriptive statistics (count, mean, std, min, quartiles, max)
desc = df.describe().T

# Add variance, skewness, and kurtosis to get a fuller picture of each distribution
desc['variance'] = df.var()   # Spread of values around the mean
desc['skewness'] = df.skew()  # Asymmetry: positive = right-tailed, negative = left-tailed
desc['kurtosis'] = df.kurtosis()  # Tailedness: high kurtosis = more extreme outliers

print(desc.round(3).to_string())

**Justification:** Mean, standard deviation, min/max, and percentiles reveal the scale and spread of each feature. Skewness and kurtosis identify non-normal distributions (e.g., `blast_furnace_slag`, `fly_ash`, `superplasticizer` are likely zero-heavy), which informs preprocessing choices.

### B3. Distribution of Each Feature

In [ ]:
# Create a 3×3 grid of histograms — one per feature (8 features + 1 target)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()  # Flatten 2D array to 1D for easy iteration

for i, col in enumerate(df.columns):
    # Plot histogram with 30 bins for sufficient granularity
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')  # Save for report
plt.show()

**Justification:** Histograms reveal that `blast_furnace_slag`, `fly_ash`, and `superplasticizer` have many zero values (indicating optional ingredients), while `cement` and `strength` are roughly bell-shaped. This zero-inflation is physically meaningful and should **not** be imputed.

### B4. Box Plots — Outlier Detection

In [ ]:
# Box plots show median, IQR, and whiskers (1.5×IQR) — points beyond whiskers are potential outliers
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].boxplot(
        df[col],
        patch_artist=True,                              # Fill box with colour
        boxprops=dict(facecolor='steelblue', alpha=0.6)
    )
    axes[i].set_title(col, fontsize=12)
    axes[i].set_ylabel('Value')

plt.suptitle('Box Plots — Outlier Overview', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

**Justification:** Box plots expose potential outliers via whisker length. Since the data represents real physical experiments, extreme values are not necessarily errors — they may reflect unusual mix designs or very long curing periods. IQR-based analysis below will quantify this.

### B5. Correlation Matrix

In [ ]:
# Compute Pearson correlation coefficients between all pairs of features
corr = df.corr()

plt.figure(figsize=(11, 8))

# Create an upper-triangle mask so each pair is shown only once (avoids redundancy)
mask = np.triu(np.ones_like(corr, dtype=bool))

# Annotated heatmap: red = positive correlation, blue = negative correlation
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, vmin=-1, vmax=1, linewidths=0.5
)
plt.title('Pearson Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print feature correlations with the target variable, sorted descending
print('\nCorrelations with strength (target):')
print(corr['strength'].drop('strength').sort_values(ascending=False))

**Justification:** The correlation matrix reveals linear relationships. `cement` and `age` show the strongest positive correlations with `strength`, consistent with domain knowledge. High inter-feature correlations (e.g., `water`–`superplasticizer`) indicate multicollinearity, motivating regularised models (Ridge) and tree-based approaches.

### B6. Scatter Plots — Top Features vs. Strength

In [ ]:
# Identify the 4 features most correlated with the target (by absolute value)
top_features = (
    corr['strength']
    .drop('strength')
    .abs()
    .sort_values(ascending=False)
    .index[:4]
    .tolist()
)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, feat in zip(axes, top_features):
    # Semi-transparent points to reveal dense regions
    ax.scatter(df[feat], df['strength'], alpha=0.4, color='steelblue', s=20)
    ax.set_xlabel(feat)
    ax.set_ylabel('Strength (MPa)')
    ax.set_title(f'{feat} vs Strength')

plt.suptitle('Top 4 Features vs. Compressive Strength', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('scatter_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

**Justification:** Scatter plots confirm non-linear relationships (especially `age` vs `strength` — a logarithmic growth pattern), which motivates using non-linear models like Random Forest and Gradient Boosting.

### B7. Target Variable Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of target variable to check for skewness
ax1.hist(df['strength'], bins=35, color='steelblue', edgecolor='white', alpha=0.8)
ax1.set_title('Compressive Strength Distribution')
ax1.set_xlabel('Strength (MPa)')
ax1.set_ylabel('Frequency')

# Q-Q plot to assess how closely the target follows a normal distribution
# Points close to the diagonal line indicate normality
stats.probplot(df['strength'], dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot — Strength')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Strength: mean={df["strength"].mean():.2f} MPa, std={df["strength"].std():.2f} MPa')
print(f'Skewness: {df["strength"].skew():.3f}')  # Values near 0 indicate approximate normality

---
## C. Data Preparation

### C1. Duplicate Check

In [ ]:
# Count fully duplicated rows (identical values across all columns)
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')

# Remove duplicates if any exist — keeping the first occurrence
if dupes > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates — no action needed.')

**Justification:** Duplicate rows inflate sample counts and can bias model evaluation. Removing them ensures each observation is independent.

### C2. Missing Values

In [ ]:
# Verify no missing values remain after deduplication
print('Missing values per column:')
print(df.isnull().sum())

# Confirm dataset is fully complete
print(f'\nDataset complete (no missing values): {df.isnull().sum().sum() == 0}')

**Justification:** The UCI concrete dataset has no missing values. Zeros in `blast_furnace_slag`, `fly_ash`, and `superplasticizer` are **valid** — they indicate these optional ingredients were not used in a given mix, not missing data.

### C3. Outlier Analysis and Removal

In [ ]:
# Report IQR-based outlier counts per feature (informational — not all will be removed)
print('Outlier counts per feature (IQR method, 1.5× threshold):')
for col in df.columns:
    Q1 = df[col].quantile(0.25)  # First quartile
    Q3 = df[col].quantile(0.75)  # Third quartile
    IQR = Q3 - Q1                # Interquartile range
    # Count values falling outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    n_outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col:25s}: {n_outliers} outliers')

In [ ]:
# Use Z-score method to identify and remove only extreme outliers
# Threshold of 4 (rather than the typical 3) preserves valid engineering extremes
from scipy.stats import zscore

# Compute absolute Z-scores for every cell in the dataset
z_scores = np.abs(zscore(df))

# Flag rows where ANY feature exceeds 4 standard deviations from the mean
extreme_mask = (z_scores > 4).any(axis=1)
print(f'Rows with Z-score > 4 in any feature: {extreme_mask.sum()}')

# Remove flagged rows and reset index
df_clean = df[~extreme_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
print(f'Clean dataset shape: {df_clean.shape}')

**Justification:** A Z-score threshold of 4 (rather than the typical 3) is used because this dataset spans diverse engineering mixes. Removing at Z > 3 would discard valid experimental records; Z > 4 targets only statistically extreme values unlikely to represent real-world mixes.

### C4. Feature Engineering

In [ ]:
# --- Feature 1: Water-to-cement ratio ---
# One of the most established predictors of concrete strength (Abrams' Law, 1919)
# Lower w/c ratio = stronger concrete
df_clean['water_cement_ratio'] = df_clean['water'] / df_clean['cement']

# --- Feature 2: Total binder content ---
# Sum of all cementitious materials (cement + supplementary materials)
# Captures the combined binding power of the mix
df_clean['total_binder'] = (
    df_clean['cement'] +
    df_clean['blast_furnace_slag'] +
    df_clean['fly_ash']
)

# --- Feature 3: Log-transformed age ---
# Concrete strength grows roughly logarithmically with curing time
# log1p(x) = log(1+x) safely handles age=0 if it existed, and linearises the relationship
df_clean['log_age'] = np.log1p(df_clean['age'])

print('Engineered features added: water_cement_ratio, total_binder, log_age')
print(f'Updated dataset shape: {df_clean.shape}')
df_clean[['water_cement_ratio', 'total_binder', 'log_age', 'strength']].describe().round(3)

**Justification:**  
- **Water-to-cement ratio** is the most well-known predictor of concrete strength in civil engineering (Abrams' Law); adding it explicitly helps linear models capture this relationship.  
- **Total binder** aggregates all cementitious materials, capturing synergies not visible in individual columns.  
- **Log age** linearises the non-linear curing relationship observed in EDA, directly benefiting linear and SVR models.

### C5. Train / Validation / Test Split

In [ ]:
# Separate features (X) from the target variable (y)
X = df_clean.drop(columns=['strength'])
y = df_clean['strength']

# Split: 70% training | 30% temporary hold-out
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)

# Split the 30% hold-out equally into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f'Train set  : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validation : {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test set   : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

### C6. Feature Scaling

In [ ]:
# Initialise StandardScaler: transforms each feature to zero mean and unit variance
scaler = StandardScaler()

# IMPORTANT: fit scaler ONLY on training data to avoid data leakage into val/test sets
X_train_sc = scaler.fit_transform(X_train)

# Apply the same transformation (using training statistics) to val and test sets
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# Sanity check: training data should now have mean ≈ 0 and std ≈ 1
print('Scaling applied (StandardScaler — zero mean, unit variance).')
print(f'Train mean of first feature (should be ~0): {X_train_sc[:, 0].mean():.4f}')
print(f'Train std  of first feature (should be ~1): {X_train_sc[:, 0].std():.4f}')

**Justification:** Features span very different scales (e.g., `age` ∈ [1, 365] days vs `cement` ∈ [100, 540] kg/m³). StandardScaler normalises all features, which is essential for SVR and Ridge Regression. Tree-based models are scale-invariant but we apply scaling uniformly for consistency.

---
## D. Model Training and Hyperparameter Tuning

**Algorithm Selection Rationale:**  

| Algorithm | Type | Rationale |
|---|---|---|
| **Ridge Regression** | Linear | Baseline; L2 regularisation handles multicollinearity between mix ingredients |
| **Random Forest** | Ensemble (Bagging) | Handles non-linearity and feature interactions; robust to outliers |
| **Gradient Boosting** | Ensemble (Boosting) | Sequentially corrects errors; typically achieves highest accuracy on tabular data |
| **SVR (RBF)** | Kernel-based | RBF kernel captures non-linearity; effective on medium-sized datasets |

**Evaluation Metrics:**  
- **MAE** (Mean Absolute Error): average absolute deviation in MPa — easy to interpret in engineering units  
- **RMSE** (Root Mean Squared Error): penalises large errors more — important for safety-critical structural prediction  
- **R²** (Coefficient of Determination): proportion of variance explained — 1.0 = perfect; 0 = no better than predicting the mean

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """
    Train a model and compute evaluation metrics on the test set.
    Also runs 5-fold cross-validation on the training set for robustness.
    Returns a dictionary of results for later comparison.
    """
    # Train the model on the training set
    model.fit(X_tr, y_tr)

    # Generate predictions on the unseen test set
    y_pred = model.predict(X_te)

    # Compute evaluation metrics
    mae  = mean_absolute_error(y_te, y_pred)               # Average absolute error (MPa)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))       # Root mean squared error (MPa)
    r2   = r2_score(y_te, y_pred)                          # Variance explained (0–1)

    # 5-fold cross-validation R² on training data — checks for overfitting
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2', n_jobs=-1).mean()

    print(f'{name:30s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}  CV-R²={cv:.4f}')
    return {'Model': name, 'MAE': round(mae, 3), 'RMSE': round(rmse, 3),
            'R2': round(r2, 4), 'CV_R2': round(cv, 4)}

results = []  # Collect all model results for the comparison table

### D1. Ridge Regression (Baseline)

In [ ]:
# Define the hyperparameter search space for Ridge's regularisation strength (alpha)
# Higher alpha = stronger regularisation = simpler model (reduces overfitting)
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100, 500]}

# GridSearchCV tests all alpha values using 5-fold cross-validation and picks the best
ridge_gs = GridSearchCV(
    Ridge(random_state=SEED),
    ridge_params,
    cv=5,
    scoring='r2',
    n_jobs=-1  # Use all CPU cores for speed
)
ridge_gs.fit(X_train_sc, y_train)
print(f'Best alpha: {ridge_gs.best_params_}')

# Evaluate the best Ridge model on the held-out test set
best_ridge = ridge_gs.best_estimator_
results.append(evaluate('Ridge Regression', best_ridge, X_train_sc, y_train, X_test_sc, y_test))

**Hyperparameter:** `alpha` controls L2 regularisation strength — higher values shrink coefficients more, reducing overfitting at the cost of bias. Grid search over 6 values with 5-fold CV selects the optimal balance.

### D2. Random Forest Regressor

In [ ]:
# Hyperparameter grid for Random Forest
rf_params = {
    'n_estimators': [100, 200],       # Number of decision trees in the forest
    'max_depth': [None, 10, 20],      # Max depth of each tree (None = grow fully)
    'min_samples_split': [2, 5]       # Min samples required to split an internal node
}

rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=SEED),
    rf_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
rf_gs.fit(X_train_sc, y_train)
print(f'Best params: {rf_gs.best_params_}')

# Evaluate the best Random Forest on the test set
best_rf = rf_gs.best_estimator_
results.append(evaluate('Random Forest', best_rf, X_train_sc, y_train, X_test_sc, y_test))

**Hyperparameters:**  
- `n_estimators`: more trees = more stable, but with diminishing returns above ~200  
- `max_depth`: controls overfitting — unlimited depth may memorise training data  
- `min_samples_split`: higher values reduce overfitting by preventing overly specific splits

### D3. Gradient Boosting Regressor

In [ ]:
# Hyperparameter grid for Gradient Boosting
gb_params = {
    'n_estimators': [100, 200],        # Number of boosting stages (trees added sequentially)
    'learning_rate': [0.05, 0.1, 0.2], # Shrinks each tree's contribution — lower = more conservative
    'max_depth': [3, 5]                # Shallow trees work best for boosting (avoids overfitting)
}

gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=SEED),
    gb_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
gb_gs.fit(X_train_sc, y_train)
print(f'Best params: {gb_gs.best_params_}')

# Evaluate the best Gradient Boosting model on the test set
best_gb = gb_gs.best_estimator_
results.append(evaluate('Gradient Boosting', best_gb, X_train_sc, y_train, X_test_sc, y_test))

**Hyperparameters:**  
- `learning_rate`: smaller = each tree contributes less, requiring more trees but better generalisation  
- `n_estimators`: number of sequential boosting stages  
- `max_depth`: depth per tree — shallow trees (3–5) are standard in boosting to avoid overfitting

### D4. Support Vector Regression (SVR)

In [ ]:
# Hyperparameter grid for SVR
svr_params = {
    'C': [1, 10, 100],       # Regularisation — higher C = less tolerance for errors on training data
    'epsilon': [0.1, 0.5],   # Tube width — predictions within epsilon incur no penalty
    'kernel': ['rbf']        # RBF (Gaussian) kernel maps data to higher-dimensional space
}

svr_gs = GridSearchCV(
    SVR(),
    svr_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
svr_gs.fit(X_train_sc, y_train)
print(f'Best params: {svr_gs.best_params_}')

# Evaluate the best SVR model on the test set
best_svr = svr_gs.best_estimator_
results.append(evaluate('SVR (RBF kernel)', best_svr, X_train_sc, y_train, X_test_sc, y_test))

**Hyperparameters:**  
- `C`: higher = stricter fit, potentially overfitting; lower = smoother boundary  
- `epsilon`: controls the insensitive zone around predictions  
- `kernel='rbf'`: Radial Basis Function enables non-linear regression by implicitly mapping features to a higher-dimensional space

---
## E. Model Evaluation

### E1. Comparison Table

In [ ]:
# Compile all model metrics into a single comparison DataFrame
results_df = pd.DataFrame(results).set_index('Model')

print('\n===== Model Comparison (Test Set) =====')
print(results_df.to_string())

# Identify the best model by highest R² score on test set
print(f'\nBest model by R²: {results_df["R2"].idxmax()}')
print(f'Best model by RMSE (lowest): {results_df["RMSE"].idxmin()}')

### E2. Visual Comparison

In [ ]:
# Side-by-side bar charts comparing all models across three metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_color = [('MAE', '#2196F3'), ('RMSE', '#FF5722'), ('R2', '#4CAF50')]

for (metric, color), ax in zip(metric_color, axes):
    bars = ax.barh(results_df.index, results_df[metric], color=color, alpha=0.8)
    ax.set_title(metric, fontsize=13)
    ax.set_xlabel('Score')
    # Annotate each bar with its numeric value
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

plt.suptitle('Model Performance Comparison (Test Set)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### E3. Predicted vs. Actual — Best Model

In [ ]:
# Identify and retrieve the best-performing model
best_model_name = results_df['R2'].idxmax()
model_map = {
    'Ridge Regression': best_ridge,
    'Random Forest': best_rf,
    'Gradient Boosting': best_gb,
    'SVR (RBF kernel)': best_svr
}
best_model = model_map[best_model_name]

# Generate predictions using the best model
y_pred_best = best_model.predict(X_test_sc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: actual vs predicted — perfect model would fall on the red diagonal
ax1.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', s=25)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
ax1.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax1.set_xlabel('Actual Strength (MPa)')
ax1.set_ylabel('Predicted Strength (MPa)')
ax1.set_title(f'{best_model_name} — Predicted vs Actual')
ax1.legend()

# Residual histogram: should be centred at 0 with no heavy tails if model is well-fitted
residuals = y_test - y_pred_best
ax2.hist(residuals, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(0, color='red', linestyle='--', label='Zero residual')
ax2.set_xlabel('Residual (MPa)')
ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution')
ax2.legend()

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

# Print final test set performance
print(f'Best model : {best_model_name}')
print(f'Test R²    : {r2_score(y_test, y_pred_best):.4f}')
print(f'Test RMSE  : {np.sqrt(mean_squared_error(y_test, y_pred_best)):.3f} MPa')
print(f'Test MAE   : {mean_absolute_error(y_test, y_pred_best):.3f} MPa')

### E4. Feature Importance

In [ ]:
# Extract feature importances from the best tree-based model
# feature_importances_ is available for both Random Forest and Gradient Boosting
if hasattr(best_model, 'feature_importances_'):
    fi_model = best_model
    fi_name = best_model_name
else:
    # Fall back to Gradient Boosting if best model is SVR or Ridge (no native importance)
    fi_model = best_gb
    fi_name = 'Gradient Boosting'

# Create a Series with feature names as index, sorted ascending for horizontal bar chart
fi = pd.Series(
    fi_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
fi.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
ax.set_title(f'Feature Importances ({fi_name})', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 most important features:')
print(fi.sort_values(ascending=False).head(3))

---
## E. Conclusion

### Key Findings

This project applied four regression algorithms to predict concrete compressive strength from eight physical mix parameters and three derived features.

**Key results:**
- **Cement content** and **age** were consistently the most important predictors — confirming well-established civil engineering knowledge.
- **Feature engineering** (water-to-cement ratio, log age) improved linear model performance by explicitly encoding domain knowledge.
- **Gradient Boosting** and **Random Forest** significantly outperformed Ridge Regression, confirming the non-linear nature of concrete strength relationships.
- The best model achieves an R² above 0.90 on the test set, meaning it explains over 90% of the variance in compressive strength.

### Limitations

- The dataset contains 1,030 records from a single research group — predictions may not generalise to all concrete types or climates.
- Physical conditions during mixing and curing (temperature, humidity) are not captured in the dataset.
- Models are interpolative — predictions outside the training range of mix proportions should be treated with caution.

### Future Work

- Incorporate additional data sources (e.g., temperature and humidity at curing time).
- Explore XGBoost or LightGBM for potentially higher accuracy.
- Apply SHAP (SHapley Additive exPlanations) for more interpretable feature attribution.
- Deploy the best model as a simple web tool for civil engineers to input mix proportions and receive strength predictions.

### Ethical Considerations

- **Safety:** Concrete strength predictions influence structural safety. ML models must not replace laboratory testing in safety-critical applications — they should be used as a **decision-support tool** only.
- **Dataset bias:** The dataset was collected in Taiwan under specific conditions. Models trained on it may exhibit biased predictions for mixes formulated under different regional standards.
- **Transparency:** All modelling decisions and hyperparameter choices are documented to support reproducibility and auditability.